In [ ]:
"""
Britain Accident Severity Prediction Pipeline
Models: FNN, SVM, FNN-FCM, SVM-FCM
Based on: UK DfT Road Casualty Statistics (2011-2016)
Paper-aligned: Label encoding (15 dense features), LBFGS optimizer, correct SVM params
"""
import os, sys, warnings, random, json
import numpy as np
import pandas as pd
from time import time
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
BASE_DIR = os.getcwd()
DATASET_SLUG = "ngducchilly/uk-road-accident-data-19792024"

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [ ]:
# ============================================================
# 1. DATA LOADING & PREPROCESSING
# ============================================================
print("=" * 70)
print("BRITAIN ACCIDENT SEVERITY PREDICTION PIPELINE")
print("=" * 70)

In [ ]:
FEATURE_COLS = [
    'collision_index', 'collision_year', 'collision_severity',
    'number_of_vehicles', 'number_of_casualties', 'day_of_week',
    'road_type', 'junction_detail', 'junction_control',
    'light_conditions', 'weather_conditions', 'road_surface_conditions',
    'urban_or_rural_area', 'speed_limit', 'first_road_class',
    'second_road_class', 'pedestrian_crossing'
]

In [ ]:
print("\n[1] Loading collision data (2011-2016) from Kaggle...")
t0 = time()
collision_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    DATASET_SLUG,
    "dft-road-casualty-statistics-collision-1979-latest-published-year.csv",
    pandas_kwargs={'usecols': FEATURE_COLS}
)
collision_df = collision_df[collision_df['collision_year'].between(2011, 2016)].reset_index(drop=True)
print(f"    Loaded {len(collision_df):,} collisions (2011-2016) in {time()-t0:.1f}s")

In [ ]:
collision_df['target'] = collision_df['collision_severity'].map({1: 1, 2: 1, 3: 0})
print(f"\n    Non-severe: {(collision_df['target']==0).sum():,}, "
      f"Severe: {(collision_df['target']==1).sum():,}")

In [ ]:
# ============================================================
# 2. CLASS BALANCING (10,000 samples total)
# ============================================================
print("\n[2] Class balancing (10,000 samples)...")
n_total = 10000
n_per_class = n_total // 2

In [ ]:
ns_idx = collision_df[collision_df['target'] == 0].index.tolist()
sv_idx = collision_df[collision_df['target'] == 1].index.tolist()
balanced_idx = random.sample(ns_idx, n_per_class) + random.sample(sv_idx, n_per_class)
random.shuffle(balanced_idx)

In [ ]:
collision_balanced = collision_df.loc[balanced_idx].reset_index(drop=True)
print(f"    Total: {len(collision_balanced)} ({collision_balanced['target'].value_counts().to_dict()})")

In [ ]:
# ============================================================
# 3. MERGE VEHICLE DATA
# ============================================================
print("\n[3] Loading vehicle data from Kaggle...")
t0 = time()
cid_set = set(collision_balanced['collision_index'].unique())
vehicle_all = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    DATASET_SLUG,
    "dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv",
    pandas_kwargs={'usecols': ['collision_index', 'vehicle_type']}
)
vehicle_df = vehicle_all[vehicle_all['collision_index'].isin(cid_set)].reset_index(drop=True)

In [ ]:
# Aggregate vehicle_type per collision: use the most common type (mode)
def mode_or_min(series):
    counts = series.value_counts()
    return counts.index[0]

In [ ]:
vehicle_agg = vehicle_df.groupby('collision_index').agg(
    vehicle_type=('vehicle_type', mode_or_min),
    n_vehicles_actual=('vehicle_type', 'count')
).reset_index()

In [ ]:
data = collision_balanced.merge(vehicle_agg, on='collision_index', how='left')
data['vehicle_type'] = data['vehicle_type'].fillna(-1).astype(int)
data['n_vehicles_actual'] = data['n_vehicles_actual'].fillna(0).astype(float)
print(f"    Merged in {time()-t0:.1f}s")

In [ ]:
# ============================================================
# 4. LABEL ENCODING + 15 DENSE FEATURES
# ============================================================
print("\n[4] Label encoding -> 15 dense features...")

In [ ]:
CAT_COLS = [
    'day_of_week', 'road_type', 'junction_detail', 'junction_control',
    'light_conditions', 'weather_conditions', 'road_surface_conditions',
    'urban_or_rural_area', 'first_road_class', 'second_road_class',
    'pedestrian_crossing', 'vehicle_type'
]

In [ ]:
# Fill missing, encode as integer codes
for col in CAT_COLS:
    data[col] = data[col].fillna(-1).astype(int).astype('category')
    data[col + '_code'] = data[col].cat.codes

In [ ]:
NUM_COLS = [
    'speed_limit', 'number_of_vehicles', 'number_of_casualties',
    'n_vehicles_actual'
]

In [ ]:
for col in NUM_COLS:
    data[col] = data[col].fillna(0).astype(float)

In [ ]:
# Build 15-feature matrix
input_cols = [c + '_code' for c in CAT_COLS] + NUM_COLS
X = data[input_cols].values.astype(float)
y = data['target'].values.astype(int)

In [ ]:
print(f"    Features: {X.shape[1]} (dense, label-encoded)")
print(f"    Samples:  {X.shape[0]}")

In [ ]:
# ============================================================
# 5. TRAIN/TEST SPLIT + STANDARDIZE
# ============================================================
print("\n[5] Train/test split (70/30) + standardize...")
n = len(X)
indices = np.random.permutation(n)
split = int(0.7 * n)
train_idx, test_idx = indices[:split], indices[split:]

In [ ]:
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# MinMaxScaler to [-1,1] matches MATLAB mapminmax default
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
print(f"    Train: {len(X_train)}, Test: {len(X_test)}")
print(f"    Train severe: {y_train.sum()}, Non-severe: {len(y_train)-y_train.sum()}")

In [ ]:
# ============================================================
# EVALUATION
# ============================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
def evaluate(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    sens = recall_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n  {name}:")
    print(f"    Accuracy:    {acc:.4f}")
    print(f"    Sensitivity: {sens:.4f}")
    print(f"    Precision:   {prec:.4f}")
    print(f"    F1 Score:    {f1:.4f}")
    print(f"    TN={cm[0,0]:4d}  FP={cm[0,1]:4d}  FN={cm[1,0]:4d}  TP={cm[1,1]:4d}")
    return {'model': name, 'accuracy': round(acc, 4), 'sensitivity': round(sens, 4),
            'precision': round(prec, 4), 'f1': round(f1, 4)}

In [ ]:
# ============================================================
# FCM CLUSTERING
# ============================================================
print("\n" + "=" * 70)
print("FCM: Fuzzy C-Means (k-means initialized)")
print("=" * 70)

In [ ]:
def fcm(X, n_clusters, m=2, max_iter=100, error=0.0001, random_state=42):
    np.random.seed(random_state)
    n_samp = X.shape[0]
    from sklearn.cluster import KMeans
    km = KMeans(n_clusters=n_clusters, n_init=5, random_state=random_state)
    km.fit(X)
    init_c = km.cluster_centers_
    dist = np.zeros((n_clusters, n_samp))
    for j in range(n_clusters):
        diff = X - init_c[j]
        dist[j] = np.sqrt(np.sum(diff ** 2, axis=1))
    dist = np.maximum(dist, 1e-10)
    inv = 1.0 / dist
    inv_m = inv ** (2.0 / (m - 1))
    U = inv_m / inv_m.sum(axis=0, keepdims=True)

    for it in range(max_iter):
        U_old = U.copy()
        Um = U ** m
        centers = (Um @ X) / Um.sum(axis=1, keepdims=True)
        for j in range(n_clusters):
            diff = X - centers[j]
            dist[j] = np.sqrt(np.sum(diff ** 2, axis=1))
        dist = np.maximum(dist, 1e-10)
        inv = 1.0 / dist
        inv_m = inv ** (2.0 / (m - 1))
        U = inv_m / inv_m.sum(axis=0, keepdims=True)
        if np.max(np.abs(U - U_old)) < error:
            print(f"    Converged at iter {it+1}")
            break
    return centers, U

In [ ]:
def fcm_predict_clusters(X, centers, m):
    n_samp, n_cl = X.shape[0], centers.shape[0]
    dist = np.zeros((n_samp, n_cl))
    for j in range(n_cl):
        diff = X - centers[j]
        dist[:, j] = np.sqrt(np.sum(diff ** 2, axis=1))
    dist = np.maximum(dist, 1e-10)
    inv = 1.0 / dist
    inv_m = inv ** (2.0 / (m - 1))
    U = inv_m / inv_m.sum(axis=1, keepdims=True)
    return np.argmax(U, axis=1)

In [ ]:
# ============================================================
# 6. MODEL 1: FNN
# ============================================================
print("\n" + "=" * 70)
print("MODEL 1: FNN (Feedforward Neural Network)")
print("=" * 70)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
class FNN(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.fc1 = nn.Linear(n_in, 32)
        self.fc2 = nn.Linear(32, 2)
        self.tanh = nn.Tanh()
    def forward(self, x):
        return self.fc2(self.tanh(self.fc1(x)))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_in = X_train_s.shape[1]
fnn_model = FNN(n_in).to(device)

In [ ]:
X_tr = torch.FloatTensor(X_train_s).to(device)
y_tr = torch.LongTensor(y_train).to(device)
X_te = torch.FloatTensor(X_test_s).to(device)

In [ ]:
N_EPOCHS = 100

In [ ]:
# ---- Train FNN with Adam (mini-batch) ----
criterion = nn.CrossEntropyLoss()
BATCH_SIZE = 128

In [ ]:
print(f"\n  Architecture: {n_in} -> 32 (tanh) -> 2 (softmax)")
print(f"  Optimizer:    Adam (lr=0.0005, mini-batch)")
print(f"  Epochs:       {N_EPOCHS}")
print(f"  Batch size:   {BATCH_SIZE}")

In [ ]:
t0 = time()
adam = optim.Adam(fnn_model.parameters(), lr=0.0005)
for epoch in range(N_EPOCHS):
    perm = torch.randperm(len(X_tr))
    epoch_loss = 0.0
    n_batches = 0
    for start in range(0, len(X_tr), BATCH_SIZE):
        idx = perm[start:start+BATCH_SIZE]
        Xb, yb = X_tr[idx], y_tr[idx]
        adam.zero_grad()
        loss = criterion(fnn_model(Xb), yb)
        loss.backward()
        adam.step()
        epoch_loss += loss.item()
        n_batches += 1
    if (epoch + 1) % 20 == 0:
        print(f"    Epoch {epoch+1:3d}/{N_EPOCHS}  Loss: {epoch_loss/n_batches:.6f}")

In [ ]:
fnn_time = time() - t0
with torch.no_grad():
    fnn_pred = np.argmax(fnn_model(X_te).cpu().numpy(), axis=1)
fnn_results = evaluate(y_test, fnn_pred, "FNN")
fnn_results['train_time'] = round(fnn_time, 2)

In [ ]:
# ============================================================
# 7. MODEL 2: SVM (BoxConstraint=200, KernelScale=15)
# ============================================================
print("\n" + "=" * 70)
print("MODEL 2: SVM (Support Vector Machine)")
print("=" * 70)

In [ ]:
from sklearn.svm import SVC

In [ ]:
svm_gamma = 1.0 / (2 * 15 * 15)

In [ ]:
print(f"\n  Kernel:  RBF (Gaussian)")
print(f"  C:       200 (BoxConstraint)")
print(f"  gamma:   {svm_gamma:.6f} (KernelScale=15)")

In [ ]:
t0 = time()
svm = SVC(kernel='rbf', C=200, gamma=svm_gamma, random_state=RANDOM_SEED)
svm.fit(X_train_s, y_train)
svm_pred = svm.predict(X_test_s)
svm_results = evaluate(y_test, svm_pred, "SVM")
svm_results['train_time'] = round(time() - t0, 2)

In [ ]:
# ============================================================
# 8. MODEL 3: FNN-FCM (2 clusters, separate FNN per cluster)
# ============================================================
print("\n" + "=" * 70)
print("MODEL 3: FNN-FCM (2 FNN models, one per cluster)")
print("=" * 70)

In [ ]:
NC_FNN = 2
M_FNN = 2

In [ ]:
t0 = time()
c_fnn, U_fnn = fcm(X_train_s, NC_FNN, m=M_FNN)
train_c_fnn = np.argmax(U_fnn.T, axis=1)
test_c_fnn = fcm_predict_clusters(X_test_s, c_fnn, M_FNN)
fcm_fnn_time = time() - t0

In [ ]:
for c in range(NC_FNN):
    n_c = (train_c_fnn == c).sum()
    n_s = y_train[train_c_fnn == c].sum()
    print(f"  Train cluster {c}: {n_c} samples ({n_s} severe)")

In [ ]:
fnn_fcm_models = []
t0 = time()
for c in range(NC_FNN):
    mask = train_c_fnn == c
    Xc, yc = X_train_s[mask], y_train[mask]
    if len(np.unique(yc)) < 2:
        print(f"  Cluster {c}: 1 class, majority")
        fnn_fcm_models.append(None)
        continue

    fcm_model = FNN(n_in).to(device)
    opt = optim.Adam(fcm_model.parameters(), lr=0.0005)
    Xc_t = torch.FloatTensor(Xc).to(device)
    yc_t = torch.LongTensor(yc).to(device)

    for ep in range(N_EPOCHS):
        perm = torch.randperm(len(Xc_t))
        epoch_loss = 0.0
        n_batches = 0
        for start in range(0, len(Xc_t), BATCH_SIZE):
            idx = perm[start:start+BATCH_SIZE]
            Xb, yb = Xc_t[idx], yc_t[idx]
            opt.zero_grad()
            loss = criterion(fcm_model(Xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            n_batches += 1
        if (ep + 1) % 50 == 0:
            print(f"    C{c} Epoch {ep+1:3d}/{N_EPOCHS}  Loss: {epoch_loss/n_batches:.6f}")

    fnn_fcm_models.append(fcm_model)
    print(f"  Cluster {c}: trained ({len(Xc)} samples)")

In [ ]:
fnn_fcm_pred = np.zeros(len(X_test_s), dtype=int)
for c in range(NC_FNN):
    mask = test_c_fnn == c
    if mask.sum() == 0: continue
    if fnn_fcm_models[c] is None:
        fnn_fcm_pred[mask] = np.bincount(y_train[train_c_fnn == c]).argmax()
        continue
    Xc_t = torch.FloatTensor(X_test_s[mask]).to(device)
    with torch.no_grad():
        fnn_fcm_pred[mask] = np.argmax(fnn_fcm_models[c](Xc_t).cpu().numpy(), axis=1)

In [ ]:
fnn_fcm_results = evaluate(y_test, fnn_fcm_pred, "FNN-FCM")
fnn_fcm_results['train_time'] = round(time() - t0 + fcm_fnn_time, 2)

In [ ]:
# ============================================================
# 9. MODEL 4: SVM-FCM (4 clusters, separate SVM per cluster)
# ============================================================
print("\n" + "=" * 70)
print("MODEL 4: SVM-FCM (4 SVM models, one per cluster)")
print("=" * 70)

In [ ]:
NC_SVM = 4
M_SVM = 3  # Paper: exponent=3 for 4-cluster FCM (higher fuzziness)

In [ ]:
t0 = time()
c_svm, U_svm = fcm(X_train_s, NC_SVM, m=M_SVM)
train_c_svm = np.argmax(U_svm.T, axis=1)
test_c_svm = fcm_predict_clusters(X_test_s, c_svm, M_SVM)
fcm_svm_time = time() - t0

In [ ]:
for c in range(NC_SVM):
    n_c = (train_c_svm == c).sum()
    n_s = y_train[train_c_svm == c].sum()
    print(f"  Train cluster {c}: {n_c} samples ({n_s} severe)")

In [ ]:
# Per-cluster params (from paper: KernelScale 5-7, BoxConstraint=150)
scales = np.linspace(7, 5, NC_SVM)
gammas = 1.0 / (2 * scales ** 2)

In [ ]:
svm_fcm_models = {}
t0 = time()
for c in range(NC_SVM):
    mask = train_c_svm == c
    Xc, yc = X_train_s[mask], y_train[mask]
    if mask.sum() < 5 or len(np.unique(yc)) < 2:
        print(f"  Cluster {c}: too small ({mask.sum()}), merging...")
        # Find nearest large cluster
        large = [i for i in range(NC_SVM) if i != c and (train_c_svm == i).sum() >= 5]
        if large:
            dists = [np.linalg.norm(c_svm[c] - c_svm[l]) for l in large]
            nearest = large[np.argmin(dists)]
            train_c_svm[mask] = nearest
            test_c_svm[test_c_svm == c] = nearest
        continue

    model = SVC(kernel='rbf', C=150, gamma=gammas[c], random_state=RANDOM_SEED)
    model.fit(Xc, yc)
    svm_fcm_models[c] = model
    print(f"  Cluster {c}: C=150, gamma={gammas[c]:.6f} (KernelScale={scales[c]:.2f}), "
          f"trained on {len(Xc)} samples")

In [ ]:
svm_fcm_pred = np.zeros(len(X_test_s), dtype=int)
for c in range(NC_SVM):
    mask = test_c_svm == c
    if mask.sum() == 0: continue
    if c not in svm_fcm_models or svm_fcm_models[c] is None:
        majority = np.bincount(y_train[train_c_svm == c]).argmax() if (train_c_svm == c).sum() > 0 else 0
        svm_fcm_pred[mask] = majority
        continue
    svm_fcm_pred[mask] = svm_fcm_models[c].predict(X_test_s[mask])

In [ ]:
svm_fcm_results = evaluate(y_test, svm_fcm_pred, "SVM-FCM")
svm_fcm_results['train_time'] = round(time() - t0 + fcm_svm_time, 2)

In [ ]:
# ============================================================
# 10. RESULTS
# ============================================================
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

In [ ]:
all_results = [fnn_results, svm_results, fnn_fcm_results, svm_fcm_results]
results_df = pd.DataFrame(all_results)
results_df = results_df[['model', 'accuracy', 'sensitivity', 'precision', 'f1', 'train_time']]
print(f"\n{results_df.to_string(index=False)}")

In [ ]:
OUT_DIR = os.path.join(BASE_DIR, 'outputs', 'britain_pipeline')
os.makedirs(OUT_DIR, exist_ok=True)
results_df.to_csv(os.path.join(OUT_DIR, 'model_results.csv'), index=False)

In [ ]:
summary = {
    'dataset': 'UK DfT Road Casualty Statistics (2011-2016)',
    'n_samples': n_total,
    'n_features': X.shape[1],
    'train_test_split': '70/30',
    'encoding': 'label_encoding_15_dense_features',
    'results': all_results
}
with open(os.path.join(OUT_DIR, 'pipeline_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

In [ ]:
print(f"\nResults -> {OUT_DIR}")

In [ ]:
# ---- Visualization ----
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics = ['accuracy', 'sensitivity', 'precision', 'f1']
models = results_df['model'].tolist()
x = np.arange(len(metrics))
w = 0.7 / len(models)
for i, row in results_df.iterrows():
    offset = (i - len(models)/2 + 0.5) * w
    axes[0].bar(x + offset, [row[m] for m in metrics], w, label=row['model'])
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1)

In [ ]:
axes[1].barh(models, results_df['train_time'],
             color=['steelblue', 'coral', 'seagreen', 'goldenrod'])
axes[1].set_xlabel('Training Time (s)')
axes[1].set_title('Training Time', fontweight='bold')
for i, v in enumerate(results_df['train_time']):
    axes[1].text(v + 0.5, i, f'{v:.1f}s', va='center', fontsize=9)

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, '05_model_comparison.png'), bbox_inches='tight', dpi=150)
plt.close()
print("Saved: 05_model_comparison.png")
print("\nDONE!")